In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir('/content/drive/Shared drives/Spring Analytics Showdown/Challenge + Dataset/')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans

In [ ]:
farmer = pd.read_csv('farmer_details.csv')
buyback = pd.read_csv('buyback_details.csv')
loan = pd.read_csv('loan_details.csv')
planting = pd.read_csv('planting_survey.csv')

In [ ]:
#Convert date columns
loan['account_package_created_at'] = pd.to_datetime(loan['account_package_created_at'], errors='coerce')
planting['planting_date'] = pd.to_datetime(planting['planting_date'], errors='coerce')
buyback['bbk_created_at'] = pd.to_datetime(buyback['bbk_created_at'], errors='coerce')

#Standardize crop names
loan['crop_variety_name'] = loan['crop_variety_name'].str.lower().str.strip()
buyback['crop_variety'] = buyback['crop_variety'].str.lower().str.strip()
planting['crop_planted'] = planting['crop_planted'].str.lower().str.strip()

In [ ]:
farmer = farmer.rename(columns={'farmer_fea_id': 'farmer_id'})

In [ ]:
loan_plant = loan.merge(
    planting,
    left_on=['farmer_id', 'crop_variety_name'],
    right_on=['farmer_id', 'crop_planted'],
    how='left'
)

In [ ]:
df = loan_plant.merge(
    buyback,
    left_on=['farmer_id', 'crop_variety_name'],
    right_on=['farmer_id', 'crop_variety'],
    how='left',
    suffixes=('', '_bbk')
)

In [ ]:
#df = df[
    #(df['bbk_created_at'] >= df['planting_date']) &
    #(df['bbk_created_at'] <= df['planting_date'] + pd.Timedelta(days=180))
#]

In [ ]:
#df = df[
    #(df['planting_date'] >= df['account_package_created_at'])
#]

In [ ]:
df = df.sort_values(by='bbk_created_at')

df = df.drop_duplicates(
    subset=['farmer_id', 'crop_variety_name', 'planting_date'],
    keep='first'
)

In [ ]:
df = df.merge(farmer, on='farmer_id', how='left')

In [ ]:
#farmer = farmer.rename(columns={'farmer_fea_id': 'farmer_id'})

#Merge datasets
#df = farmer.merge(loan, on="farmer_id", how="left") \
           #.merge(planting, on="farmer_id", how="left") \
           #.merge(buyback, on="farmer_id", how="left")

## Data Exploration

In [ ]:
df.head()

In [ ]:
print(df.columns)

In [ ]:
df.shape

In [ ]:
df = df.rename(columns={' total_weight ': 'total_weight'})

In [ ]:
#Change dates
df['dob'] = pd.to_datetime(df['dob'], errors='coerce')
df['planting_date'] = pd.to_datetime(df['planting_date'], errors='coerce')
df['account_package_created_at'] = pd.to_datetime(df['account_package_created_at'], errors='coerce')

#Age
df['age'] = 2025 - df['dob'].dt.year

#Delay
df['planting_delay'] = (df['planting_date'] - df['account_package_created_at']).dt.days

# Convert 'Yes'/'No' columns to 1/0 for input_count calculation
for col in ['fertilizer','fungicide','gypsum','inoculant','insecticide','lime','seed_guard']:
    df[col] = df[col].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

df['input_count'] = df[['fertilizer','fungicide','gypsum','inoculant','insecticide','lime','seed_guard']].sum(axis=1)

# Convert 'total_weight' and 'package_hectares' to numeric, coercing errors
df['total_weight'] = pd.to_numeric(df['total_weight'], errors='coerce')
df['package_hectares'] = pd.to_numeric(df['package_hectares'], errors='coerce')

#Yield per hectare
df['yield_per_hectare'] = df['total_weight'] / df['package_hectares']

#Binary: Map 'Yes' to 1 and 'No' to 0
df['trained'] = df['rcvd_crop_training'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

#Drop extreme invalid values
df = df[df['total_weight'] > 0]

In [ ]:
target = 'total_weight'

num_features = [
    'age', 'number_seasons', 'package_hectares',
    'Qty_kgs_planted', 'spacing_cm_btwn_rows',
    'planting_delay', 'input_count'
]

cat_features = [
    'gender', 'association', 'region_name',
    'crop_variety_name', 'program_name',
    'more_than_one_seed_in_hole'
]

In [ ]:
from sklearn.impute import SimpleImputer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', 'passthrough') # 'passthrough' or StandardScaler() if scaling is needed
        ]), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)

In [ ]:
X = df[num_features + cat_features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42))
])

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

In [ ]:
model = rf_model.named_steps['model']
feature_names = rf_model.named_steps['preprocessor'].get_feature_names_out()

importances = pd.Series(model.feature_importances_, index=feature_names)
importances.sort_values(ascending=False).head(15)

# Input Effectiveness

In [ ]:
#Compare yields with vs without fertilizer
df.groupby('fertilizer')['total_weight'].mean()

#Inputs vs yield
input_cols = ['fertilizer','insecticide','fungicide','gypsum','inoculant','lime']

for col in input_cols:
    print(f"\n{col}")
    print(df.groupby(col)['yield_per_hectare'].mean())

In [ ]:
import statsmodels.formula.api as smf

model_interact = smf.ols(
    'yield_per_hectare ~ package_hectares + fertilizer + trained + fertilizer:trained',
    data=df
).fit()

print(model_interact.summary())

# Risk Identification

In [ ]:
threshold = df['total_weight'].quantile(0.25)

df['low_yield'] = (df['total_weight'] < threshold).astype(int)

In [ ]:
early_features = [
    'age','number_seasons','package_hectares',
    'Qty_kgs_planted','spacing_cm_btwn_rows',
    'planting_delay','input_count','trained'
]

X_early = df[early_features]
y_risk = df['low_yield']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_early, y_risk, test_size=0.2, random_state=42)

In [ ]:
risk_model = RandomForestClassifier(n_estimators=200, random_state=42)
risk_model.fit(X_train_r, y_train_r)

y_pred_r = risk_model.predict(X_test_r)

from sklearn.metrics import classification_report
print(classification_report(y_test_r, y_pred_r))

In [ ]:
risk_importance = pd.Series(risk_model.feature_importances_, index=early_features)
risk_importance.sort_values(ascending=False)

In [ ]:
seg_features = df[['yield_per_hectare','package_hectares','input_count']].dropna()

kmeans = KMeans(n_clusters=3, random_state=42)
df['segment'] = kmeans.fit_predict(seg_features)

df.groupby('segment')[['yield_per_hectare','package_hectares','input_count']].mean()